In [ ]:
# ChromaCRISPR Phase 1

## Notebook 01

# Data Inventory

Objective:

Verify that all datasets required for Phase 1 Week 2 were successfully acquired.

This notebook performs:

- Inventory of ENCODE epigenomic data
- Inventory of CRISPR datasets
- File integrity checks
- Dataset size summary
- Export of inventory report

No preprocessing is performed here.

In [ ]:
from pathlib import Path
import pandas as pd
import pyBigWig

In [ ]:
PROJECT_ROOT = Path("..")

ENCODE_DIR = PROJECT_ROOT / "data/raw/encode_k562"

CRISPR_DIR = PROJECT_ROOT / "data/raw/crispr_datasets"

OUTPUT = PROJECT_ROOT / "data/interim"

OUTPUT.mkdir(exist_ok=True)

In [ ]:
encode = []

for f in sorted(ENCODE_DIR.glob("*")):

    if f.is_file():

        encode.append(
            {
                "file": f.name,
                "extension": f.suffix,
                "size_MB": round(f.stat().st_size / 1024**2,2)
            }
        )

encode_df = pd.DataFrame(encode)

encode_df

In [ ]:
validation = []

for bw in ENCODE_DIR.glob("*.bigWig"):

    try:

        x = pyBigWig.open(str(bw))

        chroms = len(x.chroms())

        x.close()

        validation.append(
            {
                "file": bw.name,
                "status": "OK",
                "chromosomes": chroms
            }
        )

    except Exception as e:

        validation.append(
            {
                "file": bw.name,
                "status": str(e),
                "chromosomes": None
            }
        )

validation_df = pd.DataFrame(validation)

validation_df

In [ ]:
records = []

for dataset in sorted(CRISPR_DIR.iterdir()):

    if not dataset.is_dir():
        continue

    total_size = 0

    extensions = {}

    nfiles = 0

    for f in dataset.rglob("*"):

        if f.is_file():

            nfiles += 1

            total_size += f.stat().st_size

            ext = f.suffix.lower()

            extensions[ext] = extensions.get(ext,0)+1

    records.append(
        {
            "dataset": dataset.name,
            "files": nfiles,
            "size_MB": round(total_size/1024**2,2),
            "extensions": ", ".join(sorted(extensions.keys()))
        }
    )

crispr_df = pd.DataFrame(records)

crispr_df

In [ ]:
summary = pd.DataFrame(
    {
        "Category":[
            "ENCODE",
            "CRISPR datasets"
        ],
        "Datasets":[
            len(validation_df),
            len(crispr_df)
        ],
        "Total size (GB)":[
            round(
                encode_df["size_MB"].sum()/1024,
                2
            ),
            round(
                crispr_df["size_MB"].sum()/1024,
                2
            )
        ]
    }
)

summary

In [ ]:
report = []

report.append("# ChromaCRISPR Phase 1")
report.append("")
report.append("## Week 2 Data Inventory")
report.append("")
report.append(summary.to_markdown(index=False))
report.append("")
report.append("### CRISPR datasets")
report.append("")
report.append(crispr_df.to_markdown(index=False))
report.append("")
report.append("### ENCODE files")
report.append("")
report.append(encode_df.to_markdown(index=False))

with open(
    OUTPUT/"week2_inventory_summary.md",
    "w"
) as f:

    f.write("\n".join(report))

print("Inventory exported.")